In [1]:
# WORKING DIR NEED TO BE SET BEFORE IMPORTING SETTINGS
import os

os.chdir("../..")
print("Working directory set to the root of the project")

Working directory set to the root of the project


In [2]:
import folium
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, box

from src.settings import EPSG_WEB_MERCATOR, EPSG_WGS84

# Read data

In [3]:
access_point_df = pd.read_csv("src/data/C2C/c2corg-anonymized.2025-12-10.access_points.csv")
access_point_gdf = gpd.GeoDataFrame(
    access_point_df,
    geometry=[Point(xy) for xy in zip(access_point_df["lon"], access_point_df["lat"])],
    crs=EPSG_WGS84,
)
access_point_gdf.head()

,document_id,lon,lat,geometry
0,1147027,55.364058,-21.021794,POINT (55.36406 -21.02179)
1,1186548,-0.358590,42.779283,POINT (-0.35859 42.77928)
2,1587979,170.816308,-43.858069,POINT (170.81631 -43.85807)
3,952546,14.510288,46.057894,POINT (14.51029 46.05789)
4,44261,7.149976,44.202556,POINT (7.14998 44.20256)


In [4]:
area_gdf = gpd.read_file("src/data/transportdatagouv/contour-des-departements.geojson")
area_gdf = area_gdf.set_crs(EPSG_WGS84, allow_override=True)
area_gdf.head()

,code,nom,geometry
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.78024 46.18905, ..."
1,02,Aisne,"POLYGON ((3.17296 50.01131, 3.17382 50.01186, ..."
2,03,Allier,"POLYGON ((3.03207 46.79491, 3.03424 46.7908, 3..."
3,04,Alpes-de-Haute-Provence,"POLYGON ((5.67604 44.19143, 5.67817 44.19051, ..."
4,05,Hautes-Alpes,"POLYGON ((6.26057 45.12685, 6.26417 45.12641, ..."


# Analysis

In [5]:
departments = ["38", "73", "74"]

In [6]:
print(f"{len(access_point_gdf)} access points in the world")

8157 access points in the world


In [7]:
france_access_point_gdf = gpd.sjoin(
    access_point_gdf,
    area_gdf.to_crs(EPSG_WGS84),
)
print(f"{len(france_access_point_gdf)} access points in France")

3707 access points in France


In [8]:
dept_access_point_gdf = gpd.sjoin(
    access_point_gdf,
    area_gdf[area_gdf["code"].isin(departments)].to_crs(EPSG_WGS84),
)
print(f"{len(dept_access_point_gdf)} access points in departments {departments}")

1366 access points in departments ['38', '73', '74']


# Plot data

In [9]:
# Plot points with bus data far from distance_threshold m

# Add a base OSM map centered on Grenoble
m = folium.Map(
    location=[45.1885, 5.7245], zoom_start=9, tiles="OpenStreetMap", control_scale=True
)

# Add Waymarked Trails hiking layers
folium.TileLayer(tiles="WaymarkedTrails.hiking").add_to(m)

# Add department polygons
for dept in departments:
    dept_latlon_coords = [
        (lat, lon)
        for lon, lat in area_gdf[area_gdf["code"] == dept]["geometry"].values[0].exterior.coords
    ]
    folium.Polygon(
        locations=dept_latlon_coords,
        color="black",
        weight=3,
        fill_color="black",
        fill_opacity=0.25,
        fill=True,
    ).add_to(m)

# Add access point markers
folium.GeoJson(dept_access_point_gdf.to_json()).add_to(m)

m